<a href="https://colab.research.google.com/github/grabs2/BUSI_Klassifikation_Projekt_Gruppe_E/blob/main/BUSI_Klassifikation_Projekt_Gruppe_E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Brust-Ultraschall-Klassifikation (BUSI)
### 1. Zielsetzung
Dieses Projekt wendet Deep Learning auf reale medizinische Bilddaten an. Ziel ist es nicht, die Genauigkeit zu maximieren, sondern den Einfluss verschiedener Modellarchitekturen (Custom CNN vs. Transfer Learning), Data-Augmentations und Hyperparameter systematisch zu untersuchen und Fehlklassifikationen zu analysieren.

### 2. Datensatz
Verwendet wird der BUSI-Datensatz (780 Ultraschallbilder, PNG):

* benign: 437 Bilder
* malignant: 210 Bilder
* normal: 133 Bilder

Die vorhandenen Masken werden beim Laden ignoriert. Für das ressourcenschonende Training in Google Colab werden alle Bilder einheitlich auf 224 x 224 Pixel skaliert.

In [1]:
import torch
import os
from pathlib import Path

# Konntrolle, ob GPU verfügbar ist und welche verwendet wird
print(f"PyTorch Version: {torch.__version__}")
is_cuda = torch.cuda.is_available()
print(f"GPU verfügbar: {is_cuda}")

if is_cuda:
    print(f"Verwendete GPU: {torch.cuda.get_device_name(0)}")

PyTorch Version: 2.11.0+cu130
GPU verfügbar: True
Verwendete GPU: Tesla T4


In [3]:
!pip install -q kagglehub

import kagglehub
from pathlib import Path

# BUSI-Datensatz herunterladen
dataset_path = kagglehub.dataset_download("aryashah2k/breast-ultrasound-images-dataset")
print("Pfad zum Datensatz:", dataset_path)

# Ordnerstruktur prüfen und Masken herausfiltern
data_dir = Path(dataset_path)
classes = ['benign', 'malignant', 'normal']
image_counts = {}

for cls in classes:
    # Sucht alle Bilddateien im jeweiligen Klassenordner
    all_files = list(data_dir.glob(f"**/{cls}/*.png"))
    # Masken-Dateien ausschliessen
    images_only = [f for f in all_files if "mask" not in f.name.lower()]
    image_counts[cls] = len(images_only)

print("\nGefundene Originalbilder pro Klasse:")
for cls, count in image_counts.items():
    print(f"  - {cls}: {count} Bilder")

# Kontrolle Vollstänidgkeit des Datensatzes
gesamt = sum(image_counts.values())
print(f"Gesamtanzahl geladener Bilder: {gesamt}")
print("Gesamtanzahl im Datensatz: 780")

Using Colab cache for faster access to the 'breast-ultrasound-images-dataset' dataset.
Pfad zum Datensatz: /kaggle/input/breast-ultrasound-images-dataset

Gefundene Originalbilder pro Klasse:
  - benign: 437 Bilder
  - malignant: 210 Bilder
  - normal: 133 Bilder
Gesamtanzahl geladener Bilder: 780
Gesamtanzahl im Datensatz: 780


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Daten in einer Tabelle (DataFrame) strukturieren
records = []
class_to_idx = {'normal': 0, 'benign': 1, 'malignant': 2}

for cls in classes:
    files = list(data_dir.glob(f"**/{cls}/*.png"))
    for f in files:
        if "mask" not in f.name.lower():
            records.append({
                'filepath': str(f),
                'label_name': cls,
                'label': class_to_idx[cls]
            })

df = pd.DataFrame(records)

# Erster Split: 70 % Training und 30 % Rest (Val + Test)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=42
)

# Die restlichen 30% gleichmässig aufteilen (jeweils 15% Val und 15% Test)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

print(f"Bilder im Trainingsset:   {len(train_df)}")
print(f"Bilder im Validationsset: {len(val_df)}")
print(f"Bilder im Testset:        {len(test_df)}")

print("\nVerteilung im Testset zur Kontrolle:")
print(test_df['label_name'].value_counts())

### Datenaufteilung und Stratified Split

Für das Training und die Evaluation des Modells wurde der BUSI-Datensatz im Verhältnis 70 % Training, 15 % Validierung und 15 % Test aufgeteilt. Da der Datensatz ein deutliches Klassenungleichgewicht aufweist (437 benign, 210 malignant, 133 normal), wurde anstelle einer rein zufälligen Verteilung ein Stratified Split (geschichtete Aufteilung) gewählt. Der Stratified Split erzwingt, dass die relativen Klassenanteile in allen drei Teilmengen identisch zum Gesamtdatensatz bleiben. Dies stellt sicher, dass:

* das Modell auf einer repräsentativen Datenbasis validiert und getestet wird,
* die errechneten Metriken (z. B. Sensitivität pro Klasse) aussagekräftig und vergleichbar bleiben,
* Verzerrungen (*Evaluation Bias*) durch Zufallseffekte bei der Datenaufteilung vermieden werden.